# GX_03 — Who Leaves the Bank, and Where? (Exploring the Data with Spark SQL)

**The question this notebook answers:** *Which customers leave the bank, and what do they have in common?*

### What this notebook is about
GX_02 built one clean customer table, `gold_churn_features`, with the original columns plus helpful groups (age group, number of products, engagement, …). Here we use it to describe the customers who left. The goal is a short list of clear findings with real numbers that management can act on, and a list of ideas for the model to test in Phase 4.

In this notebook we:
1. **Measure** the starting point: how many customers leave.
2. **Compare** churn across every customer characteristic, one at a time (Spark SQL), and compare stayers with leavers on the number columns (DataFrame API).
3. **Test** which differences are real and how big they are, with Spark's statistics tools (MLlib `ChiSquareTest` and `Correlation`).
4. **Combine** characteristics, to see whether a pattern still holds when another factor is fixed.
5. **Summarise** four findings, each with one chart for the slides, plus the hypotheses for the model.

### What this notebook uses and produces
| | |
|---|---|
| **Reads** | `gold_churn_features` (10,000 customers) |
| **Writes** | No new tables. The output is the findings, the charts and the hypotheses for Phase 4. |

### Two rules for reading the results
- **We use all 10,000 customers.** The `split` column (train / validation / test) is only for building models. Here we only count customers in groups; we do not fit any model, so the test customers stay unused by the model.
- **A pattern is not a cause.** If older customers leave more, that does not prove age *makes* them leave. Things we cannot see (service quality, a competitor's offer, the local economy) may explain it. We describe patterns; the bank decides what to try.

## Step 0 — Preparing the notebook

**What we do:** load the shared helper code, read the gold table and keep it in memory.

**Why:**
- The table is the one GX_02 built, so every number here matches what the model will see later.
- **Caching:** we query this small table about 25 times. `cache()` keeps it in memory after the first read, so Spark does not re-read it each time.
- **Temporary view:** `createOrReplaceTempView("customers")` gives the same data a name we can use in Spark SQL. The DataFrame and the SQL view are the same data; we use both styles below.
- **One extra group, only for this notebook:** salary is not grouped in the gold table, because GX_01 already suggested it carries no signal. To test that properly we split it into five bands of €40,000. Salaries are spread evenly, so these bands also hold about 2,000 customers each.

In [ ]:
# Make the shared helpers in src/ importable (this notebook lives in notebooks/).
import os
import sys

REPO_ROOT = os.path.abspath("..")
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.stat import ChiSquareTest, Correlation
from pyspark.sql import functions as F

from src.config import GOLD_CHURN

# One colour palette for every chart
COLOR_MAIN = "#27ae60"    # general bars / active customers
COLOR_DARK = "#0b5345"    # highlight / inactive customers
COLOR_GREY = "#7f8c8d"    # reference line (all customers)

print("Spark version:", spark.version)

In [ ]:
salary_band = (
    F.when(F.col("EstimatedSalary") < 40_000, "1. <40k")
    .when(F.col("EstimatedSalary") < 80_000, "2. 40-80k")
    .when(F.col("EstimatedSalary") < 120_000, "3. 80-120k")
    .when(F.col("EstimatedSalary") < 160_000, "4. 120-160k")
    .otherwise("5. 160k+")
)

customers = (
    spark.table(GOLD_CHURN)
    .withColumn("SalaryBand", salary_band)   # only for this notebook (see above)
    .cache()
)
customers.createOrReplaceTempView("customers")

print(f"{GOLD_CHURN}: {customers.count():,} customers, {len(customers.columns)} columns")

**Result:** **10,000 customers** and **23 columns**: the 22 gold columns plus the salary band.

**What we do next:** define two small helpers, so the same calculation and the same chart style are written once and reused:
- `churn_rate_by(...)` — customers, leavers and churn rate for any grouping (DataFrame API).
- `plot_rates(...)` — a bar chart of churn rates with the number of customers on each bar and the overall rate as a dashed line.

**Why show the number of customers on each bar:** a rate from 60 customers is much less reliable than a rate from 5,000. The reader should always see both.

In [ ]:
def churn_rate_by(df, *group_cols):
    # Customers, leavers and churn rate (%) per group - the core number of this notebook.
    return (
        df.groupBy(*group_cols)
        .agg(
            F.count("*").alias("customers"),
            F.sum("Exited").alias("churned"),
            F.round(F.avg("Exited") * 100, 1).alias("churn_rate_pct"),
        )
        .orderBy(*group_cols)
    )


def short_label(value):
    # Group labels start with a number so they sort correctly ("1. 18-30"); charts drop it.
    return str(value).split(". ", 1)[-1]


def plot_rates(ax, labels, rates, sizes, title, color=COLOR_MAIN):
    # Bar chart of churn rates, with n per bar and the overall rate as a dashed line.
    bars = ax.bar([short_label(x) for x in labels], rates, color=color)
    for bar, rate, n in zip(bars, rates, sizes):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f"{rate:.1f}%\n(n={n:,})", ha="center", va="bottom", fontsize=9)
    ax.axhline(OVERALL_RATE, color=COLOR_GREY, linestyle="--", linewidth=1,
               label=f"All customers ({OVERALL_RATE:.1f}%)")
    ax.set_ylim(0, max(rates) * 1.25 + 5)
    ax.set_ylabel("Churn rate (%)")
    ax.set_title(title)
    ax.legend(loc="upper left", fontsize=9)

## Step 1 — The starting point: how many customers leave?

**What we do:** count stayers and leavers with Spark SQL. The share of each group uses a **window function** (`SUM(COUNT(*)) OVER ()`): it adds up the counts of all groups, so each row can be divided by the total without a second query.

**Why:** every comparison in this notebook is measured against this number. A group is only "high risk" if it leaves more often than the average customer.

In [ ]:
overall = spark.sql('''
    SELECT Exited,
           COUNT(*)                                           AS customers,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM customers
    GROUP BY Exited
    ORDER BY Exited
''')
display(overall)

OVERALL_RATE = customers.agg(F.avg("Exited")).first()[0] * 100
print(f"Overall churn rate: {OVERALL_RATE:.1f}%")

**Result:** **2,037 of 10,000 customers left (20.4%)**, and 7,963 stayed (79.6%). So about **1 in 5** customers leaves.

From here on, **20.4% is the reference line**: a group above it leaves more than the average customer, a group below it leaves less.

## Step 2 — One characteristic at a time (Spark SQL)

**What we do:** for each customer characteristic, compute the churn rate of every group (for example France, Germany, Spain). One SQL query per characteristic, all stacked into one table with `UNION ALL`, so everything can be read side by side.

We add two helper columns:
- `vs_average_pts` — how many percentage points the group is above (+) or below (−) the 20.4% average.
- `small_group` — true if the group has fewer than 100 customers. Its rate can swing a lot by chance, so we read it with care.

**Why compare rates, not counts:** France has the most leavers simply because it has the most customers. The *rate* (leavers ÷ customers in the group) is what tells us whether a group is riskier.

In [ ]:
FACTORS = [
    "Geography", "Gender", "AgeGroup", "NumOfProducts", "IsActiveMember", "HasCrCard",
    "Engagement", "ZeroBalance", "BalanceGroup", "CreditScoreGroup", "TenureGroup", "SalaryBand",
]

one_factor_sql = " UNION ALL ".join(
    f'''
    SELECT {i} AS factor_order, '{f}' AS factor, CAST({f} AS STRING) AS grp,
           COUNT(*) AS customers, SUM(Exited) AS churned,
           ROUND(AVG(Exited) * 100, 1) AS churn_rate_pct
    FROM customers
    GROUP BY {f}'''
    for i, f in enumerate(FACTORS)
)

by_factor = spark.sql(f'''
    SELECT factor, grp, customers, churned, churn_rate_pct,
           ROUND(churn_rate_pct - {OVERALL_RATE}, 1) AS vs_average_pts,
           customers < 100                            AS small_group
    FROM ({one_factor_sql})
    ORDER BY factor_order, grp
''')
display(by_factor)

**Result:** some characteristics split customers very clearly, others not at all.

**Big differences**
- **Number of products:** 2 products **7.6%**, 1 product **27.7%**, 3 products **82.7%**, 4 products **100%** (all 60 left; a small group). Not a straight line: 2 products is the safe spot, and both fewer and more products mean more leaving.
- **Age:** **7.5%** at 18–30, 12.1% at 31–40, 34.0% at 41–50, **56.2%** at 51–60, then back down to 24.8% at 60+.
- **Country:** Germany **32.4%**, about double France (16.2%) and Spain (16.7%).
- **Activity:** inactive members **26.9%**, active members **14.3%**.
- **Engagement** (active + credit card + 2 or more products, 0 to 3): **34.5% → 29.3% → 15.9% → 9.1%**. The more ties to the bank, the less a customer leaves.
- **Gender:** women **25.1%**, men **16.5%**.
- **Balance:** customers with an empty account leave *less* (13.8%) than customers with money in it (24.1%).

**No real difference (all close to 20%)**
- Credit card: 20.2% with, 20.8% without.
- Tenure: 21.2% / 20.6% / 19.5% across the three groups.
- Credit score: 19.5%–21.2% in the four groups from 500 up; only the small group under 500 is a little higher (23.7%).
- Salary: 19.8%–21.6% across all five bands.

The `small_group` flags are on the 4-product group and the "<50k" balance group (75 customers). We do not build conclusions on those alone.

## Step 3 — Stayers vs leavers on the number columns (DataFrame API)

**What we do:** the same kind of question, asked the other way round and written with the **DataFrame API** instead of SQL: for each number column, what is the average and the middle value (median) of stayers, and of leavers?

**How the code works:** we first reshape the table so that each row is one *customer + column + value* (`explode`). Then one `groupBy` computes everything for all six columns in a single Spark job, instead of one job per column.

**Why both average and median:** an average can be pulled up or down by a few extreme customers. The median is the customer exactly in the middle, so it is not affected by extremes. If both tell the same story, the difference is solid.

In [ ]:
NUMBER_COLS = ["Age", "Balance", "NumOfProducts", "CreditScore", "Tenure", "EstimatedSalary"]

long_format = customers.select(
    "Exited",
    F.explode(F.array(*[
        F.struct(F.lit(c).alias("column"), F.col(c).cast("double").alias("value"))
        for c in NUMBER_COLS
    ])).alias("x"),
).select("Exited", "x.column", "x.value")

stayed, left = F.col("Exited") == 0, F.col("Exited") == 1
display(
    long_format.groupBy("column")
    .agg(
        F.round(F.avg(F.when(stayed, F.col("value"))), 2).alias("stayers_avg"),
        F.round(F.avg(F.when(left, F.col("value"))), 2).alias("leavers_avg"),
        F.round(F.percentile_approx(F.when(stayed, F.col("value")), 0.5), 1).alias("stayers_median"),
        F.round(F.percentile_approx(F.when(left, F.col("value")), 0.5), 1).alias("leavers_median"),
    )
    .orderBy(F.array_position(F.array(*[F.lit(c) for c in NUMBER_COLS]), F.col("column")))
)

**Result:**
- **Age** is the clearest difference: leavers are on average **44.8** years old, stayers **37.4** (medians 45 vs 36).
- **Balance:** leavers hold more money (average **€91,109** vs €72,745; median €109,349 vs €92,053). Many stayers have an empty account, which pulls their numbers down.
- **Number of products:** almost the same average (1.48 for leavers vs 1.54 for stayers), even though products turned out to be the strongest driver in Step 2. The median shows a little more (1 vs 2), but neither number can show that leavers sit at *both* ends: 1 product, or 3–4. Averages cannot see a "middle is safe" pattern; the group rates in Step 2 could.
- **Credit score, tenure and salary:** practically the same for both groups (average 645 vs 652, 4.9 vs 5.0 years, €101,466 vs €99,738). This confirms Step 2.

## Step 4 — Are the differences real, and how big are they? (MLlib statistics)

A gap between two churn rates can appear just by chance, especially in small groups. Spark's statistics library (MLlib) gives us two tools to check.

### 4.1 Chi-square test, with effect size

**What we do:** run MLlib's `ChiSquareTest` on every grouping from Step 2 against `Exited`.

**What the test does, in plain words:** it assumes that the characteristic makes *no* difference (for example, that every country has the same 20.4% churn), and calculates how surprising our real numbers would be if that were true. That surprise is the **p-value**:
- **p below 0.05** → the gap is very unlikely to be chance. We call it *real*.
- **p of 0.05 or more** → the gap could easily be chance.

**Why we also need an effect size:** with 10,000 customers, even a tiny gap of half a point can get a very small p-value. "Real" does not mean "big". So for each characteristic we also compute **Cramér's V**, which measures *how strongly* it is linked to leaving, from 0 (no link) to 1 (it decides everything). A rough guide: below 0.1 negligible, 0.1–0.3 small, 0.3–0.5 medium, above 0.5 large. With a yes/no outcome like `Exited`, V = √(chi-square ÷ number of customers).

**How the code works:** the test needs numbers, so text groups ("France", "1. 18-30") are first turned into numbers with `StringIndexer`. Then all groupings are packed into one vector column (`VectorAssembler`), and one call tests them all.

In [ ]:
GROUP_COLS = [
    "ProductsFlag", "AgeGroup", "Engagement", "Geography", "IsActiveMember", "BalanceGroup",
    "ZeroBalance", "Gender", "CreditScoreGroup", "SalaryBand", "TenureGroup", "HasCrCard",
]
text_cols = [c for c, t in customers.select(GROUP_COLS).dtypes if t == "string"]
test_cols = [f"{c}_idx" if c in text_cols else c for c in GROUP_COLS]

indexed = (
    StringIndexer(inputCols=text_cols, outputCols=[f"{c}_idx" for c in text_cols])
    .fit(customers)
    .transform(customers)
)
group_vectors = VectorAssembler(inputCols=test_cols, outputCol="groups").transform(
    indexed.select(*[F.col(c).cast("double") for c in test_cols], F.col("Exited").cast("double"))
)

chi = ChiSquareTest.test(group_vectors, "groups", "Exited", flatten=True).collect()
n_customers = customers.count()


def strength(v, p):
    if p >= 0.05:
        return "none (could be chance)"
    return "negligible" if v < 0.1 else "small" if v < 0.3 else "medium" if v < 0.5 else "large"


chi_rows = []
for r in chi:
    v = (r.statistic / n_customers) ** 0.5
    chi_rows.append((
        GROUP_COLS[r.featureIndex], int(r.degreesOfFreedom) + 1, round(float(r.statistic), 1),
        "< 0.001" if r.pValue < 0.001 else f"{r.pValue:.3f}", round(v, 3), strength(v, r.pValue),
    ))

chi_table = spark.createDataFrame(
    chi_rows, ["characteristic", "groups", "chi_square", "p_value", "cramers_v", "link_strength"]
).orderBy(F.desc("cramers_v"))
display(chi_table)

**Result:** 8 of the 12 characteristics are clearly linked to leaving; 4 are not.

| Link strength | Characteristics (Cramér's V) |
|---|---|
| **Medium** | Products (**0.39**), Age group (**0.36**) |
| **Small** | Engagement (0.21), Country (0.17), Activity (0.16), Balance group (0.13), Zero balance (0.12), Gender (0.11) |
| **None (could be chance)** | Credit score group (p = 0.13), Tenure group (p = 0.25), Credit card (p = 0.48), Salary band (p = 0.65) |

Gender shows why the effect size matters: its p-value is tiny (below 0.001), so the gap is certainly real, but the link is small (0.11), weaker than seven other characteristics. "Real" and "important" are two different questions.

**What we do next:** draw this ranking as a chart. It is the single best summary of "what matters".

In [ ]:
ranking = chi_table.orderBy("cramers_v").toPandas()   # 12 rows - safe to bring to the driver
real = ranking["link_strength"] != "none (could be chance)"

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(ranking["characteristic"], ranking["cramers_v"],
        color=[COLOR_DARK if r else COLOR_GREY for r in real])
for i, v in enumerate(ranking["cramers_v"]):
    ax.text(v + 0.005, i, f"{v:.2f}", va="center", fontsize=9)
ax.axvline(0.1, color=COLOR_GREY, linestyle=":", linewidth=1)
ax.set_xlabel("Strength of link with leaving (Cramér's V)")
ax.set_title("What is linked to leaving? Products and age matter most\n(grey = no real difference)")
plt.tight_layout()
plt.show()

**Result:** the chart shows the same ranking: **products and age stand far ahead**, then engagement, country, activity, balance and gender. Credit score, tenure, salary and credit card are at or near zero (left of the dotted 0.1 line).

### 4.2 Correlation

**What we do:** compute the correlation between every number column and `Exited`, and between the columns themselves, with MLlib's `Correlation`. We use two kinds:
- **Pearson** measures a *straight-line* link: as one value goes up, does the other go up (or down) steadily?
- **Spearman** only looks at the *order* of the values, so it also catches links that go up steadily but not in a straight line.

Both run from −1 (perfectly opposite) through 0 (no link) to +1 (perfectly together).

**Why:** two reasons.
1. A second check of Step 4.1, for the number columns.
2. **Finding columns that say the same thing.** If two model inputs are very strongly correlated (above about 0.7), the model gets the same information twice, and it becomes hard to tell which one matters. We note any such pair for Phase 4.

In [ ]:
CORR_COLS = [
    "Exited", "Age", "IsActiveMember", "Balance", "ZeroBalance", "LogBalanceToSalary",
    "NumOfProducts", "Engagement", "Gender_female", "CreditScore", "Tenure", "EstimatedSalary", "HasCrCard",
]
corr_input = customers.withColumn("Gender_female", (F.col("Gender") == "Female").cast("int"))
corr_vectors = VectorAssembler(inputCols=CORR_COLS, outputCol="numbers").transform(
    corr_input.select(*[F.col(c).cast("double") for c in CORR_COLS])
)

pearson = Correlation.corr(corr_vectors, "numbers", "pearson").first()[0].toArray()
spearman = Correlation.corr(corr_vectors, "numbers", "spearman").first()[0].toArray()

# (a) Each column against Exited
with_exited = spark.createDataFrame(
    [(CORR_COLS[j], round(float(pearson[0][j]), 3), round(float(spearman[0][j]), 3))
     for j in range(1, len(CORR_COLS))],
    ["column", "pearson_with_exited", "spearman_with_exited"],
).orderBy(F.desc(F.abs("pearson_with_exited")))
display(with_exited)

**Result (a) — each column against leaving:**
- **Age** has the strongest link (**+0.29**, Spearman +0.32): older customers leave more.
- Then **engagement** (−0.21), **activity** (−0.16, active members leave less), **zero balance** (−0.12), **balance** (+0.12) and **gender** (+0.11, women leave more). All small.
- **Number of products is close to zero (Pearson −0.05, Spearman −0.13)**, even though Step 4.1 ranked products as the *strongest* characteristic. This is the most useful lesson of this step: correlation only sees links that go steadily in one direction. Products go *down* from 1 to 2 and then sharply *up* from 2 to 4, so the two halves cancel out. **A model that only draws straight lines (like logistic regression) will underuse this column unless we help it**, for example with the 1 / 2 / 3+ groups.
- Credit score, tenure, salary and credit card: all between −0.03 and +0.01, i.e. nothing.

**What we do next:** look for pairs of input columns that carry the same information.

In [ ]:
# (b) Pairs of input columns (Exited excluded) with a strong link, |r| >= 0.5
pairs = [
    (CORR_COLS[i], CORR_COLS[j], round(float(pearson[i][j]), 3), round(float(spearman[i][j]), 3))
    for i in range(1, len(CORR_COLS)) for j in range(i + 1, len(CORR_COLS))
    if max(abs(pearson[i][j]), abs(spearman[i][j])) >= 0.5
]
display(
    spark.createDataFrame(pairs, ["column_1", "column_2", "pearson", "spearman"])
    .orderBy(F.desc(F.abs("pearson")))
)

**Result (b) — columns that overlap:**
- **Balance and ZeroBalance: −0.92** (Spearman −0.85). Expected, because ZeroBalance is built from balance. Both are useful (the zero group behaves differently), but a linear model will struggle to separate their effects.
- **Balance and LogBalanceToSalary: +0.65** (Spearman +0.82), and **ZeroBalance and LogBalanceToSalary: −0.64** (−0.85). Also expected: a zero balance means a zero ratio.
- **Engagement and its three parts** (activity +0.60, products +0.56, credit card +0.53). Expected, since Engagement is their sum.

No pair of *original* columns is strongly linked; the overlaps all come from columns we built ourselves in GX_02. **For Phase 4:** tree models are not bothered by this. For logistic regression we keep regularisation on, and we should not read its coefficients for Balance / ZeroBalance / Engagement one by one.

## Step 5 — Combining characteristics

Looking at one characteristic at a time can mislead: two characteristics can *look* important only because they come together. Here we fix one characteristic and check whether the other still makes a difference.

### 5.1 Country × balance: is Germany really riskier, or is it a balance effect?

**What we do:** churn rate by country, split into customers with an empty account and customers with money in it (Spark SQL).

**Why:** Step 2 showed that customers with money in their account leave more (24.1% vs 13.8%). If German customers all have money in their accounts, "Germany" might just be a disguised "has a balance".

In [ ]:
country_balance = spark.sql('''
    SELECT Geography,
           CASE WHEN ZeroBalance = 1 THEN 'zero balance' ELSE 'has a balance' END AS balance,
           COUNT(*)                    AS customers,
           ROUND(AVG(Exited) * 100, 1) AS churn_rate_pct
    FROM customers
    GROUP BY Geography, ZeroBalance
    ORDER BY Geography, balance
''')
display(country_balance)

**Result:** the suspicion is right that the two are tangled, but Germany still stands out.
- **Germany has no customers with an empty account at all.** All 2,509 German customers hold money with the bank.
- So the fair comparison is customers *with a balance* only: **Germany 32.4%**, France **18.2%**, Spain **19.6%**. Germany is still about **13–14 points higher**.
- In France and Spain, empty accounts leave less (13.9% and 13.6%), as in the whole bank.

**Conclusion:** the Germany effect is not just a balance effect; it survives the fair comparison. But the model cannot fully separate "German" from "has a balance", because there is no German customer with an empty account to compare with. We say this openly in the defence.

### 5.2 Age × activity

**What we do:** churn rate for every age group, split into active and inactive members. Written with the DataFrame API: `pivot` turns the two activity values into two side-by-side columns.

**Why:** age and activity were both clear drivers in Step 2. If they *add up*, the bank can find a very high-risk group by combining them.

In [ ]:
age_activity = (
    customers.groupBy("AgeGroup")
    .pivot("IsActiveMember", [0, 1])
    .agg(F.count("Exited").alias("customers"), F.round(F.avg("Exited") * 100, 1).alias("churn_pct"))
    .select(
        "AgeGroup",
        F.col("0_customers").alias("inactive_customers"), F.col("0_churn_pct").alias("inactive_churn_pct"),
        F.col("1_customers").alias("active_customers"), F.col("1_churn_pct").alias("active_churn_pct"),
    )
    .orderBy("AgeGroup")
)
display(age_activity)

**Result:** the two do not just add up, they **multiply**.

| Age | Inactive | Active | Gap |
|---|---|---|---|
| 18–30 | 10.1% | 5.1% | 5 points |
| 31–40 | 14.9% | 9.2% | 6 points |
| 41–50 | 41.6% | 25.4% | 16 points |
| **51–60** | **85.7%** | 34.7% | **51 points** |
| **60+** | **82.0%** (89 customers) | 11.2% | **71 points** |

For young customers, activity hardly matters. From about 50 onwards, it decides almost everything: **an inactive customer over 50 leaves in more than 8 out of 10 cases, an active one in 1 to 3 out of 10.** The 60+ inactive group is small (89 customers), but the 51–60 inactive group (336 customers) shows the same thing.

So the age effect in Step 2 (56% at 51–60, falling to 25% at 60+) mostly reflects **inactive older customers**. Older customers who stay active are much safer, and the 60+ group is mostly active (375 of 464).

### 5.3 Products × activity

**What we do:** the same check for the strongest driver: does the "2 products is safe" pattern hold for both active and inactive members?

In [ ]:
display(
    customers.groupBy("ProductsFlag")
    .pivot("IsActiveMember", [0, 1])
    .agg(F.round(F.avg("Exited") * 100, 1))
    .withColumnRenamed("0", "inactive_churn_pct")
    .withColumnRenamed("1", "active_churn_pct")
    .orderBy("ProductsFlag")
)

**Result:** yes, the product pattern holds whatever the activity.
- **2 products** is the safe spot for both: 9.9% (inactive) and 5.6% (active).
- **1 product:** 36.7% inactive, 18.9% active.
- **3 or more products:** 90.2% inactive, **80.3% even when active**. For these customers, being active does not help.

The safest customer in the bank is an active member with 2 products (5.6%); the riskiest product group is anyone with 3 or more.

### 5.4 Gender: does the gap disappear when country or age is fixed?

**What we do:** churn rate of women and men within each country and within each age group (Spark SQL).

**Why:** Step 2 found women leave more (25.1% vs 16.5%). Gender is a **sensitive characteristic**: if the model uses it, it may treat customers differently because of their gender. Before Phase 4 decides that, we check whether the gap is real or just comes from women being, for example, more often German or older in this data.

In [ ]:
display(spark.sql('''
    SELECT 'Country' AS fixed, Geography AS grp,
           ROUND(AVG(CASE WHEN Gender = 'Female' THEN Exited END) * 100, 1) AS women_churn_pct,
           ROUND(AVG(CASE WHEN Gender = 'Male'   THEN Exited END) * 100, 1) AS men_churn_pct
    FROM customers GROUP BY Geography
    UNION ALL
    SELECT 'Age group', AgeGroup,
           ROUND(AVG(CASE WHEN Gender = 'Female' THEN Exited END) * 100, 1),
           ROUND(AVG(CASE WHEN Gender = 'Male'   THEN Exited END) * 100, 1)
    FROM customers GROUP BY AgeGroup
    ORDER BY fixed DESC, grp
'''))

**Result:** the gap stays in **every** country and **every** age group.
- By country: France 20.3% vs 12.7%, Germany 37.6% vs 27.8%, Spain 21.2% vs 13.1% (women vs men).
- By age: women leave more in all five groups, from 9.9% vs 5.6% (18–30) to 65.2% vs 47.5% (51–60).

**Conclusion:** the gender gap is not explained by country or age. We report it, but whether the model *should* use gender is a fairness decision, not something this analysis settles. **Phase 4 trains the model with and without gender** and compares; if the model barely loses accuracy without it, we recommend leaving it out.

### 5.5 The highest-risk groups in each country

**What we do:** combine country, age group and activity, and rank the combinations inside each country by churn rate. This uses a **window function**: `RANK() OVER (PARTITION BY Geography ORDER BY churn_rate_pct DESC)` ranks the groups separately within each country, in one query. We keep only groups with at least 100 customers, so the rates are stable enough to act on.

**Why:** this is the list a retention team could use directly: "in each country, call these customers first".

In [ ]:
display(spark.sql('''
    WITH groups AS (
        SELECT Geography, AgeGroup,
               CASE WHEN IsActiveMember = 1 THEN 'active' ELSE 'inactive' END AS activity,
               COUNT(*)                    AS customers,
               ROUND(AVG(Exited) * 100, 1) AS churn_rate_pct
        FROM customers
        GROUP BY Geography, AgeGroup, IsActiveMember
        HAVING COUNT(*) >= 100
    ),
    ranked AS (
        SELECT *, RANK() OVER (PARTITION BY Geography ORDER BY churn_rate_pct DESC) AS risk_rank
        FROM groups
    )
    SELECT * FROM ranked
    WHERE risk_rank <= 3
    ORDER BY Geography, risk_rank
'''))

**Result:** the same groups top the list in every country:
1. **Inactive, 51–60:** **88.5%** in Germany (131 customers) and **83.8%** in France (142). In Spain this group has fewer than 100 customers, so it is left out.
2. **Inactive, 41–50:** **55.9%** in Germany (370), 34.4% in France (553), and 37.3% in Spain (303), where it is the top group.
3. Third place goes to **active** 51–60 year-olds (47.3% Germany, 32.9% France) — even active customers in this age band leave above average.

A retention campaign aimed at **inactive customers over 40** would reach the riskiest groups in all three countries.

## Step 6 — The four findings, each with one chart

These are the findings for the slides. Each has one sentence and one chart. All numbers come from the steps above.

### Finding 1 — Customers with 2 products rarely leave; customers with 3 or more almost all leave

**Sentence for the slide:** *2 products is the safe spot (7.6% churn). Single-product customers leave at 27.7%, and almost every customer with 3 or 4 products leaves.*

In [ ]:
products = churn_rate_by(customers, "NumOfProducts").toPandas()   # 4 rows

fig, ax = plt.subplots(figsize=(8, 5))
plot_rates(ax, products["NumOfProducts"], products["churn_rate_pct"], products["customers"],
           "Churn rate by number of products")
ax.set_xlabel("Number of products")
plt.tight_layout()
plt.show()

**Result:** 1 product **27.7%** (5,084 customers), 2 products **7.6%** (4,590), 3 products **82.7%** (266), 4 products **100%** (60).

**What the bank could do:** offer a well-chosen second product to single-product customers (half of all customers), and find out what goes wrong for customers with 3 or more products. This data does not say why they leave; that needs a closer look by the bank.

### Finding 2 — Inactive customers over 50 almost all leave

**Sentence for the slide:** *Among customers aged 51–60, 86% of inactive members leave, against 35% of active ones. For young customers, activity hardly matters.*

In [ ]:
age_pd = age_activity.toPandas()   # 5 rows
x = range(len(age_pd))
width = 0.4

fig, ax = plt.subplots(figsize=(9, 5))
bars_in = ax.bar([i - width / 2 for i in x], age_pd["inactive_churn_pct"], width, color=COLOR_DARK, label="Inactive")
bars_ac = ax.bar([i + width / 2 for i in x], age_pd["active_churn_pct"], width, color=COLOR_MAIN, label="Active")
for bars in (bars_in, bars_ac):
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f"{bar.get_height():.0f}%", ha="center", va="bottom", fontsize=9)
ax.axhline(OVERALL_RATE, color=COLOR_GREY, linestyle="--", linewidth=1, label=f"All customers ({OVERALL_RATE:.1f}%)")
ax.set_xticks(list(x))
ax.set_xticklabels([short_label(a) for a in age_pd["AgeGroup"]])
ax.set_xlabel("Age group")
ax.set_ylabel("Churn rate (%)")
ax.set_ylim(0, 105)
ax.set_title("Churn rate by age group: inactive vs active members")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

**Result:** the gap between inactive and active members grows with age: 5 points at 18–30, **51 points at 51–60** (85.7% vs 34.7%) and 71 points at 60+ (82.0% vs 11.2%, on only 89 inactive customers).

**What the bank could do:** a re-engagement campaign for inactive customers over 40, starting with those over 50. Getting an older customer active again is linked to a much lower churn rate (we cannot prove it *causes* it, but it is the clearest lever in the data).

### Finding 3 — Germany loses customers twice as fast, and it is not just a balance effect

**Sentence for the slide:** *German customers leave at 32.4%, about double France and Spain. Comparing only customers who hold money with the bank, Germany is still 13–14 points higher.*

In [ ]:
cb = country_balance.toPandas()   # 5 rows
countries = ["France", "Germany", "Spain"]
x = range(len(countries))
width = 0.4

fig, ax = plt.subplots(figsize=(9, 5))
for offset, balance, color in [(-width / 2, "zero balance", COLOR_MAIN), (width / 2, "has a balance", COLOR_DARK)]:
    rates = [cb[(cb["Geography"] == c) & (cb["balance"] == balance)]["churn_rate_pct"] for c in countries]
    for i, r in enumerate(rates):
        if len(r):
            bar = ax.bar(i + offset, float(r.iloc[0]), width, color=color)[0]
            ax.text(bar.get_x() + width / 2, bar.get_height() + 1, f"{bar.get_height():.1f}%", ha="center", fontsize=9)
        else:
            ax.text(i + offset, 1, "no empty\naccounts", ha="center", fontsize=8, color=COLOR_GREY)
ax.axhline(OVERALL_RATE, color=COLOR_GREY, linestyle="--", linewidth=1)
ax.legend(handles=[
    Patch(color=COLOR_MAIN, label="Empty account"),
    Patch(color=COLOR_DARK, label="Has a balance"),
    Line2D([0], [0], color=COLOR_GREY, linestyle="--", label=f"All customers ({OVERALL_RATE:.1f}%)"),
], loc="upper left", fontsize=9)
ax.set_xticks(list(x))
ax.set_xticklabels(countries)
ax.set_ylabel("Churn rate (%)")
ax.set_ylim(0, 45)
ax.set_title("Churn rate by country, customers with and without a balance")
plt.tight_layout()
plt.show()

**Result:** among customers with a balance, Germany **32.4%** vs France **18.2%** and Spain **19.6%**. Germany has no empty accounts, so its bar for that group is missing.

**What the bank could do:** a dedicated retention plan for Germany. The data cannot say *why* Germany is different (competition, pricing, service); that is the first question for the bank's German team.

### Finding 4 — The more ties a customer has with the bank, the less they leave

**Sentence for the slide:** *Customers with no ties to the bank (inactive, no credit card, 1 product) leave at 34.5%; customers with all three ties leave at 9.1%, almost four times less.*

**Engagement** was built in GX_02: +1 for being an active member, +1 for having a credit card, +1 for having 2 or more products.

In [ ]:
engagement = churn_rate_by(customers, "Engagement").toPandas()   # 4 rows

fig, ax = plt.subplots(figsize=(8, 5))
plot_rates(ax, engagement["Engagement"], engagement["churn_rate_pct"], engagement["customers"],
           "Churn rate by engagement (number of ties to the bank)")
ax.set_xlabel("Engagement: active member + credit card + 2 or more products")
plt.tight_layout()
plt.show()

**Result:** **34.5%** (718 customers with 0 ties) → **29.3%** (3,274) → **15.9%** (4,176) → **9.1%** (1,832 with all 3 ties). Every extra tie lowers churn.

One caution: the credit card on its own makes no difference (Step 2), so most of this ladder comes from activity and products. Engagement is a simple way to *explain* risk to management, not a new driver on its own.

## Step 7 — Hypotheses for the model (Phase 4)

The analysis gives us concrete expectations to check when we build the model. If the model disagrees, that is worth investigating.

| # | Hypothesis | Evidence here | How Phase 4 checks it |
|---|---|---|---|
| H1 | **Products and age will be the most important inputs**, then activity, country, balance and gender. | Cramér's V ranking (Step 4.1) | Feature importances of the tree models |
| H2 | **Tree models (Random Forest, GBT) will beat logistic regression clearly.** | Products is U-shaped (correlation −0.05 despite the strongest link) and age × activity multiply (Step 5.2). Straight-line models miss both. | Compare AUC-PR of the three models |
| H3 | Logistic regression improves if it gets the **1 / 2 / 3+ product groups** and an **age × inactive** term. | Steps 4.2 and 5.2 | Retrain LR with these two extra inputs |
| H4 | **Credit score, tenure, salary and credit card add almost nothing.** | No or negligible link (Steps 2–4) | Low importance; dropping them should barely change AUC-PR |
| H5 | **Country stays important even with balance in the model.** | Germany 32.4% vs 18–20% among customers with a balance (Step 5.1) | Germany's importance / coefficient with Balance and ZeroBalance included |
| H6 | **Removing gender costs little.** | Gender is a small link (0.11), smaller than seven other characteristics | Train with and without Gender; compare AUC-PR (fairness check) |
| H7 | Balance / ZeroBalance / LogBalanceToSalary and Engagement / its parts overlap. | Correlations 0.53–0.92 (Step 4.2) | Don't read LR coefficients one by one; trees are unaffected |

## Summary

### What we did
- Read the gold table (10,000 customers) and kept it in memory for about 25 queries.
- Compared churn across **12 customer characteristics** with Spark SQL, and stayers vs leavers on the number columns with the DataFrame API.
- Tested every characteristic with MLlib's **chi-square test** and measured the size of each link (**Cramér's V**); checked **Pearson and Spearman correlations** with MLlib.
- Combined characteristics (country × balance, age × activity, products × activity, gender within country and age) and ranked the riskiest groups per country with a **window function**.

### What we learned
1. **Number of products matters most:** 2 products 7.6%, 1 product 27.7%, 3–4 products 83–100%.
2. **Inactive customers over 50 almost all leave** (82–86%); active ones of the same age leave at 11–35%. Age and activity multiply.
3. **Germany churns twice as much** (32.4%), and the gap survives a fair comparison on balance (18–20% elsewhere).
4. **Engagement ladder:** 34.5% → 29.3% → 15.9% → 9.1% as ties to the bank grow.
5. **Women leave more** (25.1% vs 16.5%) in every country and age group. Reported, and passed to Phase 4 as a fairness question.
6. **Credit score, tenure, salary and credit card make no real difference.**
7. **Correlation alone would have missed the strongest driver** (products: −0.05), because the link is U-shaped.

### Decisions for later notebooks
- The seven hypotheses in Step 7 go to Phase 4 and are checked there.
- Phase 4 trains with and without `Gender` (fairness check).
- The deck uses Findings 1–4, one chart each, plus the "what matters" ranking chart if a slide is free.

---

## Next: GX_04 — Predicting who will leave (Phase 4) and customer groups (Phase 5)

Now that we know *who* leaves, the next notebook builds a model that gives every customer a **churn risk score**, so the bank can contact the riskiest customers first.
- It starts from the GX_02 preparation steps and the fixed `split` column: learn on `train`, compare models on `val`, and open `test` only once, at the very end.
- It compares logistic regression, Random Forest and Gradient-Boosted Trees, judged on **AUC-PR**, not accuracy (always saying "stays" is already 79.6% accurate).
- It checks the hypotheses above, including the fairness test without gender.
- It then groups customers into a few named segments (KMeans), to show the bank which kind of offer fits which high-risk group.